In [0]:
use globalretail.gr_silver 

Product Table

In [0]:
drop table if exists globalretail.gr_silver.product;
create table if not exists globalretail.gr_silver.product
as
select 
	product_id,
	name as prod_name,
	category,
	brand,
	price,
	stock_quantity,
	rating,
  is_active,
	current_timestamp() AS ingestion_date
from globalretail.gr_bronze.products

In [0]:
select * from globalretail.gr_silver.product

Incremental load

In [0]:
MERGE INTO globalretail.gr_silver.product AS target
USING globalretail.gr_bronze.products AS source
ON target.product_id = source.product_id

-- ✅ Update existing records
WHEN MATCHED THEN UPDATE SET
    target.prod_name = source.name,
    target.category = source.category,
    target.brand = source.brand,
    target.price = source.price,
    target.stock_quantity = source.stock_quantity,
    target.rating = source.rating,
    target.is_active = source.is_active,
       target.ingestion_date = current_timestamp()

-- ✅ Insert new records
WHEN NOT MATCHED THEN INSERT (
    product_id,
    prod_name,
    category,
    brand,
    price,
    stock_quantity,
    rating,
    is_active
)
VALUES (
    source.product_id,
    source.name,
    source.category,
    source.brand,
    source.price,
    source.stock_quantity,
    source.rating,
    source.is_active
);

-** Transformation**
- setting negative prices to 0
- setting negative stock to zero
- rating between 0 to 15
- price category - Premium , standard, budget
- stock status calculation : out of stock, low stock, moderate stock, sufficient stock

In [0]:
CREATE OR REPLACE TABLE globalretail.gr_silver.product_transform
USING DELTA
AS
SELECT
 product_id,
 prod_name,
 brand,
 CASE 
   WHEN price < 0  then 0
   else price
   END AS price ,
  CASE 
        WHEN  stock_quantity < 0 then 0
        else  stock_quantity
    END AS  stock_quantity ,
  CASE
    WHEN rating < 0  then 0
    WHEN rating >5 then 5
    else rating
    END AS rating ,
    
  case
  WHEN price > 1000 then 'Premium'
  when price> 100 then 'Standard'
  else
  'Budget'
  end as category,

  case
  when stock_quantity = 0 then 'Out Of Stock'
  when stock_quantity < 10 then 'Low Stock'
  when stock_quantity >  50 then 'High Stock'
  else 'Sufficient Stock'
  end as stock_status,
current_timestamp() AS ingestion_date

FROM globalretail.gr_silver.product


In [0]:
select * from globalretail.gr_silver.product_transform